In [ ]:
import json
import os
import random
import time

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

from modelos_scratch import MultiScaleCNN, TinyYOLOStyle, SharedTrunkMultiTask, CNNWithAttention, PyramidCNN
from dataset_class import MultiTaskObjectDetectionDataset, collate_fn


def compute_attr_acc(preds, targets):
    correct = (preds == targets).sum().item()
    total = len(targets)
    return correct / total if total > 0 else 0


def compute_iou_torch(boxes1, boxes2):
    # boxes1: [N, 4], boxes2: [M, 4]
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2 - inter
    iou = inter / union
    return iou

def train_one_model(model, model_name, train_loader, val_loader, device, num_epochs=60, early_stop_patience=5):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    history = []
    best_val_loss = float('inf')
    no_improve_epochs = 0  # contador para épocas sem melhoria

    # Parâmetros para avaliação de detecções
    iou_threshold = 0.5
    confidence_threshold = 0.6

    for epoch in range(num_epochs):
        model.train()
        total_loss, total_cls, total_box, total_attr = 0, 0, 0, 0
        batch_times = []
        total_batches = len(train_loader)
        print(f"Epoch {epoch + 1}/{num_epochs} - Total Batches: {total_batches}")

        for batch_idx, (images, targets, attrs) in enumerate(train_loader, start=1):
            start_time = time.time()

            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            attrs = [{k: v.to(device) for k, v in a.items()} for a in attrs]

            optimizer.zero_grad()
            loss_dict = model(images, targets, attrs)
            loss = sum(v for v in loss_dict.values())

            loss.backward()
            optimizer.step()

            batch_time = time.time() - start_time
            batch_times.append(batch_time)
            print(f"Epoch {epoch + 1}, Batch {batch_idx}/{total_batches}: Time = {batch_time:.2f} s")

            total_loss += loss.item()
            total_cls += loss_dict.get("classification_loss", 0).item()
            total_box += loss_dict.get("bbox_regression_loss", 0).item()
            total_attr += loss_dict.get("attribute_loss", 0).item()

        avg_batch_time = sum(batch_times) / len(batch_times) if batch_times else 0

        # Validação: calcular acurácia dos atributos e das detecções
        model.eval()
        with torch.no_grad():
            all_w, all_s, all_t = [], [], []
            all_wp, all_sp, all_tp = [], [], []
            detection_correct = 0
            detection_total = 0
            detections_count = 0
            
            for images, targets, attrs in val_loader:
                if not images:
                    continue
                images = [img.to(device) for img in images]
                # Obter predições: chamamos o modelo sem targets e attrs para receber tupla (detections, atributos)
                detections, attr_out = model(images, None, None)
                # Acurácia para atributos globais
                all_wp += torch.argmax(attr_out["weather"], dim=1).cpu().tolist()
                all_sp += torch.argmax(attr_out["scene"], dim=1).cpu().tolist()
                all_tp += torch.argmax(attr_out["timeofday"], dim=1).cpu().tolist()
                all_w += [a["weather"].item() for a in attrs]
                all_s += [a["scene"].item() for a in attrs]
                all_t += [a["timeofday"].item() for a in attrs]

                # Acurácia para detecções: compara as caixas preditas com as caixas de ground truth
                for i in range(len(images)):
                    gt_boxes = targets[i]["boxes"].to(device)
                    gt_labels = targets[i]["labels"].to(device)
                    detection_total += len(gt_boxes)

                    pred_boxes = detections[i]["boxes"].to(device)
                    pred_labels = detections[i]["labels"].to(device)
                    pred_scores = detections[i]["scores"].to(device)

                    # Filtrar predições pelo threshold de confiança
                    keep = pred_scores >= confidence_threshold
                    pred_boxes = pred_boxes[keep]
                    pred_labels = pred_labels[keep]

                    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                        continue

                    ious = compute_iou_torch(gt_boxes, pred_boxes)
                    max_iou, max_idx = ious.max(dim=1)
                    matches = max_iou >= iou_threshold
                    detections_count += matches.sum().item()
                    if matches.sum().item() > 0:
                        detection_correct += (pred_labels[max_idx[matches]] == gt_labels[matches]).sum().item()

            acc_weather = compute_attr_acc(torch.tensor(all_wp), torch.tensor(all_w)) if all_w else 0
            acc_scene = compute_attr_acc(torch.tensor(all_sp), torch.tensor(all_s)) if all_s else 0
            acc_time = compute_attr_acc(torch.tensor(all_tp), torch.tensor(all_t)) if all_t else 0
            detection_accuracy = detection_correct / detection_total if detection_total > 0 else 0
            avg_detections = detections_count / len(val_loader.dataset) if len(val_loader.dataset) > 0 else 0

        history.append({
            "epoch": epoch + 1,
            "total_batches": total_batches,
            "total_loss": total_loss,
            "classification_loss": total_cls,
            "bbox_loss": total_box,
            "attribute_loss": total_attr,
            "avg_batch_time": avg_batch_time,
            "acc_weather": acc_weather,
            "acc_scene": acc_scene,
            "acc_time": acc_time,
            "detection_accuracy": detection_accuracy,
            "avg_detections": avg_detections
        })

        print(f"[{model_name}] Epoch {epoch + 1}: loss={total_loss:.4f}, avg batch time={avg_batch_time:.2f} s, "
              f"attr acc: W={acc_weather:.2%} S={acc_scene:.2%} T={acc_time:.2%}, "
              f"detection acc: {detection_accuracy:.2%}, avg detections: {avg_detections:.2f}")

        # Early Stopping: verifica se houve melhoria na loss total
        if total_loss < best_val_loss:
            best_val_loss = total_loss
            no_improve_epochs = 0
            torch.save(model, f"best_{model_name}.pth")
        else:
            no_improve_epochs += 1
            print(f"Sem melhoria na loss. Épocas sem melhoria: {no_improve_epochs}/{early_stop_patience}")
            if no_improve_epochs >= early_stop_patience:
                print("Early stopping acionado. Interrompendo treinamento.")
                break

    # Salvar histórico de treino em arquivo JSON
    os.makedirs("logs", exist_ok=True)
    with open(f"logs/training_log_{model_name}.json", "w") as f:
        json.dump(history, f, indent=2)

    return history


# Caminhos para os dados
splits = {
    "train": ("images/train", "labels/train"),
    "val": ("images/val", "labels/val")
}

# Transforms
transform = transforms.Compose([transforms.ToTensor()])

# Carregamento dos datasets e DataLoaders (usa 50% do dataset)
datasets, loaders = {}, {}
for split, (img_dir, lbl_dir) in splits.items():
    print(f"📂 Carregando dataset '{split}'...")
    ds = MultiTaskObjectDetectionDataset(img_dir, lbl_dir, transform)
    subset_size = max(1, int(len(ds) * 0.50))
    indices = random.sample(range(len(ds)), subset_size)
    ds_subset = Subset(ds, indices)
    datasets[split] = ds_subset
    loaders[split] = DataLoader(
        ds_subset, batch_size=8, shuffle=(split == "train"),
        collate_fn=collate_fn, num_workers=0
    )

with open("helpers/categories.json", "r") as f:
    category_to_label = json.load(f)
with open("helpers/weather.json", "r") as f:
    weather_to_label = json.load(f)
with open("helpers/scene.json", "r") as f:
    scene_to_label = json.load(f)
with open("helpers/timeofday.json", "r") as f:
    timeofday_to_label = json.load(f)

# Parâmetros
num_classes = len(category_to_label) + 1
num_weather = len(weather_to_label)
num_scene = len(scene_to_label)
num_time = len(timeofday_to_label)

# Solicitar ao usuário a seleção dos modelos a treinar
model_input = input("Modelos disponíveis: multiscale, tinyyolo, sharedtask, attention, pyramid\n"
                    "Digite os modelos a treinar, separados por vírgula (ou 'all' para todos): ").strip().lower()
if model_input == "all" or model_input == "":
    selected_models = ["multiscale", "tinyyolo", "sharedtask", "attention", "pyramid"]
else:
    selected_models = [m.strip() for m in model_input.split(",")]

print(f"Modelos selecionados: {selected_models}")

# Criação do dicionário de modelos (agora com os parâmetros para atributos)
all_models = {
    "multiscale": MultiScaleCNN(num_classes, num_weather, num_scene, num_time),
    "tinyyolo": TinyYOLOStyle(num_classes, num_weather, num_scene, num_time),
    "sharedtask": SharedTrunkMultiTask(num_classes, num_weather, num_scene, num_time),
    "attention": CNNWithAttention(num_classes, num_weather, num_scene, num_time),
    "pyramid": PyramidCNN(num_classes, num_weather, num_scene, num_time),
}

models = {k: v for k, v in all_models.items() if k in selected_models}

train_loader, val_loader = loaders["train"], loaders["val"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = {}
for name, model in models.items():
    print(f"🔧 Treinando modelo: {name}")
    results[name] = train_one_model(model, name, train_loader, val_loader, device, num_epochs=60)

plt.figure(figsize=(12, 6))
for name, hist in results.items():
    plt.plot([h["total_loss"] for h in hist], label=f"{name} - loss")
plt.title("Loss total por modelo")
plt.legend()
plt.grid(True)
plt.savefig("logs/loss_comparison.png")
plt.show()
